# QSVM médical : comprendre l'adaptation CPU et MerLin

Ce notebook présente les petits résultats déjà calculés. Il ne relance ni simulation quantique ni calcul de kernel.

Trois périmètres doivent rester distincts :

1. la reproduction de référence sur les embeddings médicaux contrôlés du papier, non réalisée ici ;
2. l'étude de substitution sur les pixels de PneumoniaMNIST ;
3. l'adaptation photonic avec MerLin, qui n'est pas le circuit BSP à qubits du papier.

## 1. Charger les artefacts nettoyés

Les fichiers sous `results/` contiennent uniquement de petites métriques et de la métadonnée. Les images, pixels et matrices de kernel ne sont pas inclus.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
results_path = project_root / "results"
summary_path = results_path / "curated_results.csv"
merlin_path = results_path / "merlin_q2_seed0.json"

if not summary_path.exists() or not merlin_path.exists():
    raise FileNotFoundError("Lancez ce notebook depuis la racine qsvm_medimage.")

results = pd.read_csv(summary_path)
results

## 2. Comparaison CPU à q=4

La métrique principale est le **F1 de la classe minoritaire** (`normal`). Elle combine :

- la précision : parmi les images prédites normales, combien le sont réellement ;
- le rappel : parmi les images réellement normales, combien sont détectées.

Un F1 minoritaire égal à zéro signifie qu'aucune image minoritaire n'a été correctement détectée.

In [ ]:
q4 = results.query("scope == 'open_data_surrogate' and pca_dim == 4").copy()
q4[["model", "n_seeds", "mean_test_minority_f1", "std_test_minority_f1"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    q4["model"],
    q4["mean_test_minority_f1"],
    yerr=q4["std_test_minority_f1"],
    capsize=4,
)
ax.set_ylabel("F1 minoritaire test")
ax.set_ylim(0, 1.05)
ax.set_title("PneumoniaMNIST, q=4, N=100, 10 seeds appariées")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()

Le QSVM est en moyenne légèrement au-dessus du SVM linéaire (`+0,013`), mais légèrement sous le RBF ajusté (`−0,020`). Les écarts sont petits face à la variabilité entre seeds. Cette étude ne reproduit donc pas l'avantage systématique annoncé dans le papier.

## 3. Smoke du kernel photonic MerLin

Cette expérience utilise deux composantes PCA, trois modes optiques et l'état de Fock `[1, 0, 1]`. La seed des données et celle du circuit valent toutes deux zéro, mais elles ont des rôles distincts.

In [ ]:
with merlin_path.open() as file:
    merlin = json.load(file)

merlin_metrics = pd.DataFrame(merlin["metrics"]).T
merlin_metrics.index.name = "split"
merlin_metrics

In [ ]:
ax = merlin_metrics[["accuracy", "minority_f1", "auc"]].plot.bar(
    figsize=(8, 4), rot=0
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("MerLin q=2, seed données=0, seed circuit=0")
ax.figure.tight_layout()

L'AUC test vaut 1, mais le F1 minoritaire vaut 0. Cela n'est pas contradictoire : l'AUC mesure l'ordre des scores pour tous les seuils possibles, tandis que le F1 utilise les décisions produites au seuil du SVC. Ici, le modèle classe les scores dans le bon ordre mais prédit finalement toutes les images comme `pneumonia`.

Le test ne contient que dix images, dont deux normales. Ce résultat valide le fonctionnement technique de MerLin sur CPU ; il ne démontre aucun avantage photonic.

## 4. Ce qu'il reste à conclure

- Les pixels PneumoniaMNIST et les embeddings gelés du papier ne sont pas scientifiquement équivalents.
- Le kernel MerLin et le BSP à qubits sont deux feature maps différentes.
- Les résultats actuels servent à vérifier le pipeline et à formuler les prochaines expériences, pas à revendiquer un avantage quantique.
- Toute comparaison suivante doit garder les mêmes échantillons et splits pour les modèles comparés.